In [11]:
from google.cloud import bigquery

client = bigquery.Client(project="prod-organize-arizon-4e1c0a83")

query = """
SELECT
*
FROM `prod-organize-arizon-4e1c0a83.viewers_dataset.full_program_2024`
"""

df = client.query(query).to_dataframe()

df.head()

,GEOMETRY,PCTNUM,county,canvass_perc,low_prop_perc,bipoc_perc
0,"POLYGON((-111.939185 34.675424, -111.967628 34...",YA0328,Yavapai,0.015014,0.213432,0.169738
1,"POLYGON((-111.743146 34.876266, -111.743289 34...",CN0083,Coconino,0.043357,0.152323,0.082952
2,"POLYGON((-111.743146 34.876266, -111.743289 34...",CN0083,Coconino,0.043357,0.152323,0.082952
3,"POLYGON((-112.43318 34.584265, -112.451466 34....",YA0301,Yavapai,0.006369,0.150026,0.074281
4,"POLYGON((-111.941463 34.677875, -111.928354 34...",YA0334,Yavapai,0.009511,0.177589,0.082656


In [12]:

# have to create geometry that folium can work with from bigquery friendly wkt geometry field
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.validation import make_valid  # Shapely ≥2.0

# point to the column that contains WKT, here it's called GEOMETRY
wkt_col = "GEOMETRY"

# parse WKT into shapely geometries
geoms = gpd.GeoSeries.from_wkt(df[wkt_col])

# build a GeoDataFrame and assign the correct CRS
# If WKT is already lon/lat (WGS84), use EPSG:4326. Otherwise, set the true source EPSG and then .to_crs(4326).
gdf = gpd.GeoDataFrame(df.drop(columns=["geometry"], errors="ignore"),
                       geometry=geoms,
                       crs="EPSG:4326")

In [26]:
import geopandas as gpd
import folium, branca
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Prep geometry & columns ---
gdf = gdf.copy()
gdf["geometry"] = gdf["geometry"].buffer(0)                         # fix minor invalidities
gdf["geometry"] = gdf["geometry"].simplify(0.0005, preserve_topology=True)

# ensure county is present and clean
assert "county" in gdf.columns, "Expected a 'county' column."
gdf["county"] = gdf["county"].astype(str).str.strip()

metrics = ["canvass_perc", "low_prop_perc", "bipoc_perc"]

# human-readable percentage labels for tooltips (original cols stay as 0..1 decimals)
for col in metrics:
    gdf[f"{col}_lbl"] = (gdf[col] * 100).round(1).astype(str) + "%"

# remove pct with nans
gdf = gdf.dropna(subset=metrics, how="any")

# --- County-aware renderer (recomputes color scale per selection) ---
def render_map_for(county, metric):
    # subset to county (or statewide)
    gsub = gdf if (county is None or county == "<All>") else gdf[gdf["county"] == county]
    if gsub.empty:
        raise ValueError(f"No rows for county={county}")

    # center and create map
    minx, miny, maxx, maxy = gsub.total_bounds
    m = folium.Map(
        location=[(miny + maxy) / 2, (minx + maxx) / 2],
        zoom_start=(8 if county and county != "<All>" else 6),
        tiles="CartoDB Positron",
        prefer_canvas=True,
    )

    # county-specific color scale
    vmin = float(gsub[metric].min())
    vmax = float(gsub[metric].max())
    if vmin == vmax:
        vmax = vmin + 1e-9  # avoid flat legend

    cmap = branca.colormap.linear.YlOrRd_09.scale(vmin, vmax)
    pretty = {
        "canvass_perc": "Canvassed %",
        "low_prop_perc": "Low-Propensity %",
        "bipoc_perc": "BIPOC %",
    }[metric]
    cmap.caption = f"{pretty} — {county if county and county != '<All>' else 'All AZ'} (county scale)"

    def style_fn(feat):
        v = feat["properties"].get(metric)
        return {
            "fillColor": cmap(v) if v is not None else "#ccc",
            "color": "#555",
            "weight": 0.4,
            "fillOpacity": 0.85,
        }

    tooltip = folium.GeoJsonTooltip(
        fields=["PCTNUM", f"{metric}_lbl", "county"],
        aliases=["PCTNUM", pretty, "County"],
        sticky=True,
        labels=True,
    )

    folium.GeoJson(
        data=gsub.__geo_interface__,
        name=pretty,
        style_function=style_fn,
        tooltip=tooltip,
        highlight_function=lambda f: {"weight": 1.2, "color": "#000"},
    ).add_to(m)

    # auto-fit the map to selection bounds
    m.fit_bounds([[miny, minx], [maxy, maxx]])

    cmap.add_to(m)
    return m

# --- Simple interactive controls ---
county_opts = ["<All>"] + sorted(gdf["county"].dropna().unique())
metric_opts = [("Canvassed %", "canvass_perc"),
               ("Low-Propensity %", "low_prop_perc"),
               ("BIPOC %", "bipoc_perc")]

county_dd = widgets.Dropdown(options=county_opts, value="<All>", description="County:")
metric_dd = widgets.Dropdown(options=metric_opts, value="canvass_perc", description="Metric:")
out = widgets.Output()

def refresh(_=None):
    with out:
        clear_output()
        display(render_map_for(county_dd.value, metric_dd.value))

county_dd.observe(refresh, names="value")
metric_dd.observe(refresh, names="value")

display(widgets.HBox([county_dd, metric_dd]))
refresh()
display(out)


Output()